<div align="center">

<img src="https://s3.amazonaws.com/files.pucp.edu.pe/pucp-general/img-header/logo-pucp-new.svg" alt="Pontificia Universidad Católica del Perú">

<br><br>

## Pontificia Universidad Católica del Perú  

### Deep Learning con Python  

<br>

## Semana 3 — Tarea 3

### Comparando CNN y Transfer Learning

</div>

---

Objetivo de este notebook:

- Implementar una red CNN y verificar su performance

- Aplicar Transfer Learning para mejorar la performance

---

## Contexto del caso

En esta tarea se trabajará con un conjunto de **imágenes de resonancia magnética cerebral (MRI)** utilizadas para la detección de tumores cerebrales. El dataset contiene imágenes clasificadas en cuatro categorías: **glioma**, **meningioma**, **pituitary tumor** y **no tumor**. Cada imagen representa un corte del cerebro obtenido mediante resonancia magnética, técnica ampliamente utilizada en el diagnóstico médico para identificar anomalías estructurales en el tejido cerebral.

El objetivo del ejercicio es desarrollar modelos de **clasificación de imágenes** capaces de identificar correctamente el tipo de tumor presente en la resonancia. En primer lugar, se entrenará una **red neuronal convolucional (CNN) desde cero**, permitiendo analizar cómo un modelo aprende características visuales directamente a partir de los datos. Posteriormente, se aplicará **Transfer Learning utilizando un modelo preentrenado**, lo que permitirá reutilizar características aprendidas previamente en grandes datasets de imágenes y evaluar si esta estrategia mejora el desempeño del modelo en el problema de clasificación médica.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

## Lectura y preparación del dataset

En esta sección se cargan las imágenes del dataset utilizando las utilidades de **TensorFlow/Keras**. Las imágenes se encuentran organizadas en directorios, donde cada subcarpeta representa una clase distinta del problema de clasificación (glioma, meningioma, notumor y pituitary).

Para la lectura del dataset se utiliza la función `image_dataset_from_directory`, la cual permite cargar automáticamente las imágenes, asignar etiquetas en función del nombre de las carpetas y generar un dataset listo para ser utilizado en modelos de Deep Learning.

Durante este proceso también se realiza:

- **Redimensionamiento de las imágenes** a un tamaño fijo (`IMG_SIZE`), con el objetivo de mantener consistencia en la entrada del modelo.
- **Aleatorización de los datos (shuffle)** para evitar sesgos durante el entrenamiento.
- Creación de dos conjuntos iniciales:
  - **Training dataset**, utilizado para entrenar el modelo.
  - **Testing dataset**, que posteriormente será dividido en **validación y prueba** para evaluar el desempeño del modelo.

In [ ]:
train_dir = "dataset_cnn/Training"
test_dir = "dataset_cnn/Testing"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

In [ ]:
train_ds = None
test_val_ds = None

#########################
# COMPLETAR ESTA SECCION DEL CODIGO ANTES DE EJECUTAR LAS SIGUIENTES CELDAS
# train_ds =  tf.keras.utils.image_dataset_from_directory()
# test_val_ds = tf.keras.utils.image_dataset_from_directory()
#########################

## Conversión del dataset y preparación de los conjuntos de datos

Una vez que las imágenes han sido cargadas desde los directorios utilizando `image_dataset_from_directory`, el siguiente paso consiste en **convertir el dataset en arreglos de NumPy**. Esto facilita su manipulación y permite utilizarlos directamente durante el entrenamiento de los modelos de Deep Learning.

En las siguientes celdas se recorrerá el dataset de entrenamiento (`train_ds`) y el dataset de prueba inicial (`test_val_ds`) para extraer:

- **Las imágenes**, que se almacenarán en los arreglos `X`.
- **Las etiquetas**, que se almacenarán en los arreglos `y`.

Durante este proceso cada imagen se convierte a formato NumPy mediante `.numpy()`, y posteriormente todas las observaciones se agrupan en arreglos utilizando `np.array()`.

De esta manera se generan los siguientes conjuntos:

- **X_train, y_train** → datos de entrenamiento  
- **X_test_val, y_test_val** → datos que posteriormente serán divididos en validación y prueba

Posteriormente, el conjunto `X_test_val` se divide en dos subconjuntos:

- **Test set** → utilizado para evaluar el desempeño final del modelo
- **Validation set** → utilizado durante el entrenamiento para monitorear el desempeño del modelo y evitar sobreajuste

En este caso se realiza una división simple del **50% para test y 50% para validación**.

---

## Visualización de imágenes del dataset

Finalmente, se seleccionan **9 imágenes aleatorias del conjunto de entrenamiento** con el objetivo de explorar visualmente el dataset.

Esto permite:

- verificar que las imágenes se hayan cargado correctamente  
- observar la variabilidad presente en las imágenes  
- identificar las diferentes clases del problema de clasificación

Las imágenes se muestran en una cuadrícula de **3 × 3**, donde cada imagen incluye el nombre de la clase correspondiente:

- glioma  
- meningioma  
- notumor  
- pituitary

Este paso es importante para **entender el tipo de datos con los que trabajará el modelo de Deep Learning** antes de iniciar el proceso de entrenamiento.

In [ ]:
# ESTA CELDA DIVIDE EL DATASET TRANSFORMANDOLO A OBJETOS DE NUMPY, SOLO EJECUTARLA LUEGO DE IMPLEMENTAR LA CELDA ANTERIOR
# ADEMAS SE MUESTRAN 9 IMAGENES ALEATORIAS DEL DATASET ESTUDIADO

X_train = []
y_train = []

for img, label in train_ds:
    X_train.append(img.numpy())
    y_train.append(label.numpy())

X_train = np.array(X_train)
y_train = np.array(y_train)

X_test_val = []
y_test_val = []

for img, label in test_val_ds:
    X_test_val.append(img.numpy())
    y_test_val.append(label.numpy())

X_test_val = np.array(X_test_val)
y_test_val = np.array(y_test_val)

n = len(X_test_val)

test_size = int(n * 0.5)

X_test = X_test_val[:test_size]
y_test = y_test_val[:test_size]

X_val = X_test_val[test_size:]
y_val = y_test_val[test_size:]

class_names = ["glioma", "meningioma", "notumor", "pituitary"]

plt.figure(figsize=(8,8))

indices = np.random.choice(len(X_train), 9, replace=False)

for i, idx in enumerate(indices):
    
    plt.subplot(3,3,i+1)
    plt.imshow(X_train[idx])
    plt.title(class_names[y_train[idx]])
    plt.axis("off")

plt.show()

## Normalización de las imágenes y verificación de dimensiones

Antes de entrenar los modelos de Deep Learning, es necesario realizar un paso importante de **preprocesamiento de los datos: la normalización de las imágenes**.

Las imágenes cargadas originalmente tienen valores de píxel en el rango:

0 - 255


Sin embargo, las redes neuronales suelen entrenarse de forma más eficiente cuando los valores de entrada se encuentran en un rango más pequeño. Por esta razón, se normalizan los valores de los píxeles dividiéndolos entre **255**, lo que transforma los datos al rango:

0 - 1


Esta transformación permite:

- mejorar la estabilidad del entrenamiento
- acelerar la convergencia del modelo
- evitar problemas numéricos durante la optimización

Luego de normalizar los datos, se imprimen las dimensiones de los conjuntos:

- **X_train** → conjunto de entrenamiento  
- **X_val** → conjunto de validación  
- **X_test** → conjunto de prueba  

Esto permite verificar que:

- las imágenes tengan el tamaño esperado  
- los conjuntos de datos se hayan dividido correctamente  
- el número de observaciones sea consistente para el entrenamiento y evaluación del modelo

In [ ]:
X_train = None
X_val = None
X_test = None

## Codificación de etiquetas (One-Hot Encoding)

Hasta este punto las etiquetas del dataset (`y_train`, `y_val`, `y_test`) se encuentran representadas como **valores enteros** que identifican cada clase:

- 0 → glioma  
- 1 → meningioma  
- 2 → notumor  
- 3 → pituitary  

Sin embargo, cuando se entrena un modelo de clasificación multiclase utilizando una función de pérdida como **categorical_crossentropy**, es necesario convertir estas etiquetas a **one-hot encoding**.

El **one-hot encoding** representa cada clase como un vector binario donde únicamente una posición toma el valor **1** y el resto **0**.

Ejemplo:

| Clase | Representación |
|------|------|
glioma | [1, 0, 0, 0] |
meningioma | [0, 1, 0, 0] |
notumor | [0, 0, 1, 0] |
pituitary | [0, 0, 0, 1] |

Para realizar esta transformación se utiliza la función:

tf.keras.utils.to_categorical()

donde se especifica el número total de clases del problema.

En este caso:

NUM_CLASSES = 4

Luego de realizar la codificación, es importante **verificar las dimensiones de los datos** utilizando `.shape`.

Las dimensiones esperadas son:

### Dimensiones de los conjuntos de entrada

Train shape: (5600, 224, 224, 3)
Validation shape: (800, 224, 224, 3)
Test shape: (800, 224, 224, 3)


### Dimensiones de las etiquetas codificadas

(5600, 4)
(800, 4)
(800, 4)

Esto confirma que cada observación ahora está representada por un **vector de 4 posiciones**, una por cada clase del problema de clasificación.

In [19]:
NUM_CLASSES = 4

y_train = None
y_val = None
y_test = None

## Implementación de una CNN desde cero

En esta sección se implementará una **Red Neuronal Convolucional (CNN)** para resolver el problema de clasificación de imágenes de resonancias magnéticas cerebrales.

El objetivo de esta arquitectura es **aprender automáticamente patrones visuales presentes en las imágenes**, tales como bordes, texturas y formas asociadas a los diferentes tipos de tumores.

La red que se construirá es una **CNN sencilla**, compuesta por capas convolucionales, capas de reducción de dimensionalidad (*max pooling*) y capas densas para la clasificación final.

---

### Arquitectura del modelo

El modelo debe construirse utilizando la API **`Sequential`** de Keras y debe contener las siguientes capas:

1. **Primera capa convolucional**
   - `Conv2D`
   - **16 filtros**
   - kernel **(3,3)**
   - función de activación **ReLU**
   - **padding='valid'**
   - `input_shape=(224,224,3)`

   ⚠️ Es importante que dentro de esta capa se incluya explícitamente el parámetro:

padding='valid'

Este tipo de padding indica que **no se agregan píxeles adicionales alrededor de la imagen**, por lo que el tamaño espacial se reduce después de aplicar la convolución.

2. **Primera capa de reducción de dimensionalidad**
- `MaxPooling2D`
- tamaño de ventana **(2,2)**

Esta capa reduce el tamaño espacial de los mapas de características, lo que permite disminuir el costo computacional del modelo.

---

3. **Segunda capa convolucional**
- `Conv2D`
- **24 filtros**
- kernel **(3,3)**
- activación **ReLU**
- **padding='valid'**

Nuevamente se debe especificar el parámetro:

padding='valid'


---

4. **Segunda capa de Max Pooling**
- `MaxPooling2D`
- tamaño **(2,2)**

---

5. **Flatten**

La capa `Flatten` se utiliza para **transformar los mapas de características en un vector unidimensional**, que podrá ser procesado por las capas densas.

---

6. **Capa densa**
- `Dense`
- **64 neuronas**
- activación **ReLU**

Esta capa permite aprender combinaciones no lineales de las características extraídas por las capas convolucionales.

---

7. **Dropout**
- `Dropout(0.4)`

Esta capa se utiliza para **reducir el sobreajuste (overfitting)** durante el entrenamiento, desactivando aleatoriamente el 40% de las neuronas en cada iteración.

---

8. **Capa de salida**
- `Dense`
- **4 neuronas** (una por cada clase)
- activación **softmax**

La función **softmax** permite obtener probabilidades para cada una de las cuatro clases del problema:

- glioma
- meningioma
- notumor
- pituitary

---

Una vez definida esta arquitectura, el modelo estará listo para ser **compilado y entrenado** en las siguientes secciones del notebook.

In [ ]:
model_cnn = None

## Compilación del modelo

Una vez definida la arquitectura de la red neuronal convolucional, el siguiente paso consiste en **compilar el modelo**. Durante este proceso se especifican los elementos necesarios para entrenar la red neuronal.

En particular se deben definir:

- **Optimizador (optimizer)**: algoritmo encargado de actualizar los pesos de la red durante el entrenamiento.
- **Función de pérdida (loss)**: mide qué tan lejos se encuentran las predicciones del modelo respecto a las etiquetas reales.
- **Métricas de evaluación (metrics)**: permiten monitorear el desempeño del modelo durante el entrenamiento.

Para este ejercicio se utilizarán los siguientes parámetros:

### Optimizador
Se utilizará el optimizador **Adam**, uno de los más utilizados en Deep Learning debido a su buena estabilidad y velocidad de convergencia.

El optimizador debe configurarse con un **learning rate de 0.0005**.

### Función de pérdida
Dado que se trata de un problema de **clasificación multiclase** y las etiquetas han sido convertidas a **one-hot encoding**, se utilizará la función de pérdida:

categorical_crossentropy

### Métrica de evaluación
Se utilizará la métrica:

accuracy


la cual mide la proporción de predicciones correctas realizadas por el modelo.

---

## Resumen del modelo

Luego de compilar el modelo, es importante visualizar la arquitectura completa utilizando el método:

model.summary()


Este comando permite observar:

- las capas del modelo
- la dimensión de las salidas de cada capa
- el número de parámetros entrenables
- el número total de parámetros del modelo

Este paso es útil para **verificar que la arquitectura haya sido construida correctamente antes de iniciar el entrenamiento**.


In [ ]:
### COMPILAR EL MODELO CREADO Y VISUALIZAR SU RESUMEN

## model_cnn.compile()
## model_cnn.summary()

## Definición de callbacks para el entrenamiento

Durante el entrenamiento de redes neuronales es común utilizar **callbacks**, que son funciones que se ejecutan automáticamente durante el proceso de entrenamiento y permiten controlar o ajustar el comportamiento del modelo.

En este ejercicio se utilizarán dos callbacks importantes:

- **EarlyStopping**
- **ReduceLROnPlateau**

Estos mecanismos ayudan a mejorar el entrenamiento del modelo y a evitar problemas como el **sobreajuste (overfitting)** o el estancamiento del proceso de optimización.

---

### EarlyStopping

El callback **EarlyStopping** permite **detener el entrenamiento automáticamente** cuando el modelo deja de mejorar en el conjunto de validación.

En este caso se configurará con los siguientes parámetros:

- `monitor="val_loss"`  
  El criterio de monitoreo será la **pérdida en el conjunto de validación**.

- `patience=6`  
  El entrenamiento se detendrá si la pérdida de validación **no mejora durante 6 épocas consecutivas**.

- `restore_best_weights=True`  
  Cuando el entrenamiento se detenga, el modelo recuperará automáticamente **los pesos correspondientes a la mejor época obtenida**.

- `verbose=1`  
  Permite mostrar mensajes informativos cuando el entrenamiento se detiene.

---

### ReduceLROnPlateau

El callback **ReduceLROnPlateau** permite **reducir automáticamente el learning rate** cuando el modelo deja de mejorar en el conjunto de validación.

Esto ayuda a que el optimizador pueda seguir refinando los pesos del modelo cuando el entrenamiento comienza a estancarse.

Se configurará con los siguientes parámetros:

- `monitor="val_loss"`  
  Se monitorea la pérdida del conjunto de validación.

- `factor=0.5`  
  Cuando se detecta un estancamiento, el learning rate se reduce a **la mitad**.

- `patience=3`  
  Si la pérdida de validación **no mejora durante 3 épocas**, el learning rate será reducido.

- `min_lr=1e-6`  
  Se establece un límite mínimo para evitar que el learning rate se vuelva demasiado pequeño.

- `verbose=1`  
  Muestra mensajes cuando el learning rate es reducido.

---

Estos callbacks se utilizarán posteriormente durante el entrenamiento del modelo para **mejorar la estabilidad del proceso de aprendizaje y evitar entrenamientos innecesariamente largos**.

In [ ]:
early_stop = None
reduce_lr = None

## Entrenamiento del modelo CNN

Una vez definida la arquitectura del modelo, compilado el optimizador y configurados los callbacks, el siguiente paso consiste en **entrenar la red neuronal convolucional** utilizando el conjunto de entrenamiento.

Durante el entrenamiento el modelo aprenderá a identificar patrones en las imágenes que permitan diferenciar entre las cuatro clases del problema:

- glioma  
- meningioma  
- notumor  
- pituitary  

El entrenamiento se realizará con los siguientes parámetros:

- **epochs = 30**  
  El modelo podrá entrenarse hasta un máximo de 30 épocas.

- **batch_size = 64**  
  Las imágenes se procesarán en grupos de 64 durante cada actualización de los pesos.

- **validation_data = (X_val, y_val)**  
  Se utilizará el conjunto de validación para monitorear el desempeño del modelo durante el entrenamiento.

- **callbacks = [early_stop, reduce_lr]**  
  Se utilizarán los callbacks definidos anteriormente para detener el entrenamiento si deja de mejorar y para ajustar automáticamente el learning rate cuando el proceso se estanque.

Es importante tener en cuenta que el **tiempo de entrenamiento puede variar dependiendo del hardware utilizado**. En una computadora personal con CPU el entrenamiento suele demorar aproximadamente entre:

**7 y 15 minutos**

Si se dispone de **GPU**, el tiempo de entrenamiento puede ser considerablemente menor.

Durante el entrenamiento se mostrarán en pantalla las métricas de **loss** y **accuracy** tanto para el conjunto de entrenamiento como para el conjunto de validación.

In [ ]:
history_cnn = None

## Visualización del proceso de entrenamiento

Luego de finalizar el entrenamiento del modelo, es importante **analizar cómo evolucionó el aprendizaje a lo largo de las épocas**. Para ello se utilizará el objeto `history` devuelto por la función `model.fit()`.

Este objeto contiene el registro de las métricas calculadas durante cada época de entrenamiento, tanto para el conjunto de **entrenamiento** como para el conjunto de **validación**.

En las siguientes celdas se deben generar dos gráficos:

1. **Accuracy vs Épocas**
   - Muestra cómo evoluciona la precisión del modelo durante el entrenamiento.
   - Se grafican dos curvas:
     - accuracy en entrenamiento
     - accuracy en validación

2. **Loss vs Épocas**
   - Muestra cómo evoluciona la función de pérdida durante el entrenamiento.
   - También se grafican dos curvas:
     - pérdida en entrenamiento
     - pérdida en validación

Estas gráficas permiten evaluar el comportamiento del modelo y detectar posibles problemas como:

- **Overfitting** (cuando el modelo aprende demasiado bien el entrenamiento pero no generaliza bien)
- **Underfitting** (cuando el modelo no logra aprender adecuadamente los patrones de los datos)

Analizar estas curvas es una práctica fundamental para **entender el desempeño del modelo antes de evaluar los resultados finales en el conjunto de prueba**.

In [ ]:
plt.figure(figsize=(12,5))

# VISUALIZAR AMBAS GRAFICAS DE (ACCURACY, VAL_ACCURACY) Y (LOSS, VAL_LOSS) EN UNA SOLA FIGURA
plt.subplot(1,2,1) # SUBPLOT PARA ACCURACY, VAL_ACCURACY

plt.subplot(1,2,2) # SUBPLOT PARA LOSS, VAL_LOSS


plt.show()

## Evaluación del modelo en el conjunto de prueba

Una vez finalizado el entrenamiento y analizado el comportamiento del modelo durante las épocas, el siguiente paso consiste en **evaluar el desempeño del modelo en el conjunto de prueba (test set)**.

Este conjunto de datos **no ha sido utilizado durante el entrenamiento**, por lo que permite obtener una estimación más realista de la capacidad del modelo para **generalizar a datos nuevos**.

Para ello se utilizará el método:

model.evaluate()


Este método devuelve dos métricas principales:

- **Loss en el conjunto de prueba**
- **Accuracy en el conjunto de prueba**

La métrica de **accuracy** indica el porcentaje de imágenes que el modelo clasifica correctamente.

---

## Classification Report

Además de la accuracy, es útil analizar métricas más detalladas para cada clase utilizando el **classification report** de `scikit-learn`.

El classification report permite observar tres métricas importantes para cada clase:

- **Precision**  
  Indica qué proporción de las predicciones realizadas por el modelo para una clase son correctas.

- **Recall**  
  Indica qué proporción de los ejemplos reales de una clase fueron correctamente identificados por el modelo.

- **F1-score**  
  Es una medida que combina precision y recall en un solo indicador.

También se muestra el **support**, que corresponde al número de ejemplos de cada clase presentes en el conjunto de prueba.

En este problema de clasificación se evaluarán las siguientes clases:

- glioma  
- meningioma  
- notumor  
- pituitary  

Estas métricas permiten analizar **qué tan bien el modelo reconoce cada tipo de tumor cerebral** y detectar posibles clases que resulten más difíciles de clasificar.

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

## Transfer Learning

Hasta este punto se ha entrenado una **Red Neuronal Convolucional (CNN) desde cero**, donde todos los parámetros del modelo fueron aprendidos únicamente a partir del dataset de resonancias magnéticas.

Sin embargo, entrenar redes profundas desde cero puede requerir **grandes cantidades de datos y tiempo de entrenamiento**. Para abordar este problema, se utiliza una técnica conocida como **Transfer Learning**.

El **Transfer Learning** consiste en reutilizar modelos previamente entrenados en grandes datasets de imágenes (como ImageNet) para resolver nuevos problemas de visión por computadora. Estos modelos ya han aprendido a detectar **características visuales generales**, tales como:

- bordes  
- texturas  
- patrones visuales  
- formas complejas

Estas características pueden reutilizarse para otros problemas de clasificación de imágenes, incluso si las imágenes pertenecen a dominios distintos.

En esta sección se utilizará el modelo preentrenado **MobileNetV2**, una arquitectura eficiente y ampliamente utilizada en tareas de visión por computadora. Este modelo fue entrenado originalmente sobre el dataset **ImageNet**, que contiene millones de imágenes de distintas categorías.

Al cargar este modelo preentrenado:

- se utilizarán **los pesos aprendidos en ImageNet**
- se eliminará la capa final de clasificación (`include_top=False`)
- se utilizará el modelo como **extractor de características**

A continuación, ejecute la siguiente celda para cargar el modelo base **MobileNetV2** que será utilizado para el proceso de Transfer Learning.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

## Construcción del modelo con Transfer Learning

Una vez cargado el modelo base **MobileNetV2**, el siguiente paso consiste en **construir el modelo final de clasificación** que utilizará las características extraídas por esta red preentrenada.

Recordemos que el modelo MobileNetV2 fue entrenado sobre el dataset **ImageNet**, por lo que ya ha aprendido a detectar patrones visuales complejos en imágenes. En este ejercicio utilizaremos este modelo como **extractor de características** y agregaremos nuevas capas al final para adaptarlo a nuestro problema de clasificación de tumores cerebrales.

Para implementar el modelo utilizaremos la API **`Sequential`** de Keras.

El modelo debe construirse siguiendo la siguiente estructura:

1. **Modelo base (MobileNetV2)**  
   El primer elemento del modelo debe ser el modelo preentrenado `base_model` que se cargó previamente.

2. **GlobalAveragePooling2D**  
   Esta capa se utiliza para convertir los mapas de características generados por MobileNetV2 en un vector unidimensional.  
   A diferencia de `Flatten`, esta capa reduce significativamente el número de parámetros del modelo.

3. **Capa densa intermedia**
   - `Dense`
   - **64 neuronas**
   - función de activación **ReLU**

   Esta capa permite aprender combinaciones de las características extraídas por MobileNetV2.

4. **Dropout**
   - `Dropout(0.4)`

   Esta capa se utiliza para reducir el **overfitting**, desactivando aleatoriamente el 40% de las neuronas durante el entrenamiento.

5. **Capa de salida**
   - `Dense`
   - **4 neuronas**
   - activación **softmax**

   Esta capa genera las probabilidades para cada una de las cuatro clases del problema:

   - glioma  
   - meningioma  
   - notumor  
   - pituitary  

En resumen, el modelo debe construirse utilizando `tf.keras.Sequential()` e incluir las capas en el siguiente orden:

- `base_model`
- `GlobalAveragePooling2D`
- `Dense(64, activation='relu')`
- `Dropout(0.4)`
- `Dense(4, activation='softmax')`

Una vez definida esta arquitectura, el modelo estará listo para ser **compilado y entrenado** en las siguientes secciones.

In [ ]:
model_tl = None

## Compilación del modelo de Transfer Learning

Una vez definida la arquitectura del modelo que incorpora **MobileNetV2** como extractor de características, el siguiente paso consiste en **compilar el modelo** antes de iniciar el entrenamiento.

Durante la compilación se especifican tres elementos fundamentales del proceso de aprendizaje:

### Optimizador

Se utilizará el optimizador **Adam**, ampliamente utilizado en modelos de Deep Learning debido a su buena estabilidad y velocidad de convergencia.

En este caso el optimizador debe configurarse con un **learning rate de 0.001**.

### Función de pérdida

Dado que el problema corresponde a **clasificación multiclase** y las etiquetas han sido codificadas utilizando **one-hot encoding**, se utilizará la función de pérdida:

categorical_crossentropy

### Métrica de evaluación

Para evaluar el desempeño del modelo durante el entrenamiento se utilizará la métrica:

accuracy


que indica el porcentaje de predicciones correctas realizadas por el modelo.

---

## Resumen del modelo

Luego de compilar el modelo, es recomendable visualizar la arquitectura completa utilizando el método:

model_tl.summary()


Este comando permite observar:

- la estructura completa del modelo
- las dimensiones de salida de cada capa
- el número de parámetros entrenables
- el número total de parámetros del modelo

Este paso es útil para **verificar que la arquitectura del modelo de Transfer Learning haya sido construida correctamente antes de iniciar el proceso de entrenamiento**.

In [ ]:
# model_tl.compile()
# model_tl.summary()

## Definición de callbacks

Antes de entrenar el modelo con Transfer Learning, se definirán nuevamente dos **callbacks** que ayudarán a controlar el proceso de entrenamiento y mejorar la estabilidad del aprendizaje.

### EarlyStopping

El callback **EarlyStopping** permite detener el entrenamiento automáticamente cuando el modelo deja de mejorar en el conjunto de validación.

Debe configurarse con los siguientes parámetros:

- `monitor="val_loss"` → monitorea la pérdida en el conjunto de validación  
- `patience=6` → detiene el entrenamiento si no hay mejora durante **6 épocas consecutivas**  
- `restore_best_weights=True` → restaura los pesos correspondientes a la **mejor época obtenida**  
- `verbose=1` → muestra mensajes informativos durante el entrenamiento  

### ReduceLROnPlateau

El callback **ReduceLROnPlateau** permite **reducir automáticamente el learning rate** cuando el entrenamiento deja de mejorar.

Debe configurarse con los siguientes parámetros:

- `monitor="val_loss"` → monitorea la pérdida de validación  
- `factor=0.5` → reduce el learning rate a **la mitad** cuando se detecta estancamiento  
- `patience=3` → espera **3 épocas sin mejora** antes de reducir el learning rate  
- `min_lr=1e-6` → establece un valor mínimo para el learning rate  
- `verbose=1` → muestra mensajes cuando el learning rate es reducido  

Estos callbacks se utilizarán durante el entrenamiento del modelo para **evitar sobreentrenamiento y mejorar la convergencia del modelo**.

In [ ]:
early_stop = None
reduce_lr = None

## Entrenamiento del modelo con Transfer Learning

Una vez compilado el modelo y definidos los callbacks, se procederá a **entrenar el modelo que utiliza MobileNetV2 como extractor de características**.

Para este entrenamiento se deben utilizar **exactamente los siguientes parámetros**:

- `epochs = 6`
- `batch_size = 64`
- `validation_data = (X_val, y_val)`
- `callbacks = [early_stop, reduce_lr]`

⚠️ **Es importante que el número de épocas sea únicamente 6.**  
No se deben aumentar las épocas, ya que el modelo MobileNetV2 es una red profunda y el entrenamiento puede volverse muy costoso computacionalmente.

Debido a la complejidad del modelo, **el tiempo de entrenamiento puede ser considerablemente mayor que en la CNN desde cero**.

En una computadora personal sin GPU, el entrenamiento puede tardar aproximadamente:

**entre 20 y 30 minutos**

Por esta razón, se recomienda **no modificar el número de épocas ni los parámetros indicados**.

Durante el entrenamiento se mostrarán las métricas de:

- **loss**
- **accuracy**
- **val_loss**
- **val_accuracy**

lo que permitirá monitorear el desempeño del modelo en el conjunto de entrenamiento y validación.

In [ ]:
history_tl = None 

## Visualización del entrenamiento

Una vez finalizado el entrenamiento del modelo con **Transfer Learning**, genere las gráficas que muestran la evolución del entrenamiento a lo largo de las épocas.

En particular, se deben visualizar dos gráficos:

1. **Accuracy vs Épocas**
   - accuracy en entrenamiento
   - accuracy en validación

2. **Loss vs Épocas**
   - pérdida en entrenamiento
   - pérdida en validación

Estas gráficas permiten observar el comportamiento del modelo durante el entrenamiento y verificar si el modelo **mejora correctamente o presenta signos de sobreajuste (overfitting)**.

In [ ]:
plt.figure(figsize=(12,5))

# VISUALIZAR AMBAS GRAFICAS DE (ACCURACY, VAL_ACCURACY) Y (LOSS, VAL_LOSS) EN UNA SOLA FIGURA
plt.subplot(1,2,1) # SUBPLOT PARA ACCURACY, VAL_ACCURACY

plt.subplot(1,2,2) # SUBPLOT PARA LOSS, VAL_LOSS


plt.show()

## Evaluación final del modelo con Transfer Learning

Finalmente, se evaluará el desempeño del modelo entrenado con **Transfer Learning** utilizando el conjunto de prueba (`X_test`, `y_test`).  

Para ello se utilizará el método:

model_tl.evaluate()


Este método permite calcular la **pérdida (loss)** y la **accuracy** del modelo sobre datos que **no fueron utilizados durante el entrenamiento**.

Posteriormente, se generará un **classification report** utilizando la librería `scikit-learn`. Este reporte permite analizar el desempeño del modelo para cada clase mediante las métricas:

- **Precision**
- **Recall**
- **F1-score**
- **Support**

Estas métricas permiten entender **qué tan bien el modelo identifica cada tipo de tumor cerebral**.

---

## Preguntas de análisis

Responda las siguientes preguntas en una celda de **texto o comentario dentro del notebook**:

1. **¿Qué modelo obtuvo mejor desempeño en el conjunto de prueba?**  
   Compare los resultados obtenidos con la **CNN entrenada desde cero** y el modelo basado en **Transfer Learning con MobileNetV2**.

2. **¿Por qué cree que el modelo con Transfer Learning puede obtener mejores resultados?**  
   Explique brevemente el concepto de **Transfer Learning** y cómo influye en el aprendizaje del modelo.

3. **¿Cuál de los dos modelos tiene una arquitectura más compleja o mayor número de parámetros?**  
   Justifique su respuesta basándose en el `model.summary()` observado previamente.

Estas preguntas tienen como objetivo **reflexionar sobre las diferencias entre entrenar una red desde cero y utilizar modelos preentrenados en tareas de visión por computadora**.

In [ ]:
from sklearn.metrics import classification_report
import numpy as np